<h2>Take Home Test from The Economist Group </h2>

<h3>Coding Test 2</h3>

<P>
The three worksheets (CPI; Imports-India; Exports-India; Exchange_Rates) contains the dataset for Consumer Price Index, Exchange Rate for multiple countries, along with India’s bilateral trade (import from and export to) data with all partner countries. Use this data to create a Real Effective Exchange Rate (REER) time series for India.

You need to  code the estimation of the REER in R or Python</P>

In [1]:
# Import required libraries
import pandas as pd
import numpy as np

In [2]:
# Load datasets
cpi = pd.read_csv("CPI.csv")
exchange = pd.read_csv("Exchange_Rates.csv")
exports = pd.read_csv("Exports-India.csv")
imports = pd.read_csv("Imports-India.csv")

In [3]:
# Preview data
print("CPI Data:")
print(cpi.head())

print("\nExchange Rate Data:")
print(exchange.head())

print("\nExports Data:")
print(exports.head())

print("\nImports Data:")
print(imports.head())

CPI Data:
     Geography                                         Definition  \
0  Afghanistan  The consumer price index rebased to 2010=100 b...   
1      Albania  The consumer price index rebased to 2010=100 b...   
2      Algeria  The consumer price index rebased to 2010=100 b...   
3       Angola  The consumer price index rebased to 2010=100 b...   
4     Anguilla  The consumer price index rebased to 2010=100 b...   

                                                Note   1980   1981   1982  \
0                                                NaN      –      –      –   
1                                                NaN      –      –      –   
2   As of May 2019, long term forecast (2026-2050...  5.933  6.803  7.248   
3                                                NaN      –      –      –   
4                                                NaN      –      –      –   

    1983   1984   1985    1986  ...     2011     2012     2013     2014  \
0      –      –      –       –  ...  

In [4]:
# Check structure
print("\nCPI Columns:", cpi.columns)
print("Exchange Columns:", exchange.columns)
print("Exports Columns:", exports.columns)
print("Imports Columns:", imports.columns)


CPI Columns: Index(['Geography', 'Definition', 'Note', '1980', '1981', '1982', '1983',
       '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992',
       '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001',
       '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
       '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
       '2020'],
      dtype='object')
Exchange Columns: Index(['Geography', 'Definition', 'Note', '1980', '1981', '1982', '1983',
       '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992',
       '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001',
       '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
       '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
       '2020', '2021'],
      dtype='object')
Exports Columns: Index(['Exports to', '1980 *y', '.DESC', '1980', '1981', '1982', '1983',
       '1984',

<h4>Converting all dataset to long format.</h4>
<P>Right now your data is wide (years as columns).
We need long format so we can merge later.</P>

<h3>1. CPI</h3>

In [5]:
# Rename
cpi.rename(columns={"Geography": "Country"}, inplace=True)

# Select ONLY year columns (skip first 3 columns)
cpi_year_cols = cpi.columns[3:]   # ['1980', '1981', ..., '2020']

# Melt
cpi_long = cpi.melt(id_vars="Country",
                    value_vars=cpi_year_cols,
                    var_name="Year",
                    value_name="CPI")

# Convert Year to integer
cpi_long["Year"] = cpi_long["Year"].astype(int)

# Check
print(cpi_long.head())

       Country  Year    CPI
0  Afghanistan  1980      –
1      Albania  1980      –
2      Algeria  1980  5.933
3       Angola  1980      –
4     Anguilla  1980      –


<h3>2. Exchange Rates</h3>

In [6]:
exchange.rename(columns={"Geography": "Country"}, inplace=True)

# Skip first 3 columns
exchange_year_cols = exchange.columns[3:]

exchange_long = exchange.melt(id_vars="Country",
                              value_vars=exchange_year_cols,
                              var_name="Year",
                              value_name="Exchange_Rate")

exchange_long["Year"] = exchange_long["Year"].astype(int)
print(exchange_long.head())

       Country  Year Exchange_Rate
0  Afghanistan  1980         34.37
1      Albania  1980             –
2      Algeria  1980       3.83745
3       Angola  1980             0
4     Anguilla  1980           2.7


<h3>3. Exports</h3>

In [7]:
exports.rename(columns={"Exports to": "Country"}, inplace=True)

exports_year_cols = exports.columns[3:]

exports_long = exports.melt(id_vars="Country",
                            value_vars=exports_year_cols,
                            var_name="Year",
                            value_name="Exports")

exports_long["Year"] = exports_long["Year"].astype(int)
print(exports_long.head())

   Country  Year  Exports
0       US  1980   967.03
1       UK  1980   528.75
2  Austria  1980    10.90
3  Belgium  1980      NaN
4  Denmark  1980    34.98


<h3>4. Imports</h3>

In [8]:
imports.rename(columns={"Imports from": "Country"}, inplace=True)

imports_year_cols = imports.columns[3:]

imports_long = imports.melt(id_vars="Country",
                            value_vars=imports_year_cols,
                            var_name="Year",
                            value_name="Imports")

imports_long["Year"] = imports_long["Year"].astype(int)
print(imports_long.head())

   Country  Year  Imports
0       US  1980  1865.21
1       UK  1980   953.53
2  Austria  1980    34.00
3  Belgium  1980      NaN
4  Denmark  1980    26.32


<h4>Data Cleaning</h4>

<h3>1. CPI Cleaning</h3>

In [9]:
# Replace weird values with NaN
cpi_long["CPI"] = cpi_long["CPI"].replace("–", np.nan)

# Convert to numeric
cpi_long["CPI"] = pd.to_numeric(cpi_long["CPI"], errors="coerce")

print(cpi_long.head())

       Country  Year    CPI
0  Afghanistan  1980    NaN
1      Albania  1980    NaN
2      Algeria  1980  5.933
3       Angola  1980    NaN
4     Anguilla  1980    NaN


<h3>2. Exchange Rate Cleaning</h3>

In [10]:
exchange_long["Exchange_Rate"] = exchange_long["Exchange_Rate"].replace("–", np.nan)

exchange_long["Exchange_Rate"] = pd.to_numeric(exchange_long["Exchange_Rate"], errors="coerce")

print(exchange_long.head())

       Country  Year  Exchange_Rate
0  Afghanistan  1980       34.37000
1      Albania  1980            NaN
2      Algeria  1980        3.83745
3       Angola  1980        0.00000
4     Anguilla  1980        2.70000


<h3>3. Exports Cleaning</h3>

In [11]:
exports_long["Exports"] = exports_long["Exports"].replace("–", np.nan)

exports_long["Exports"] = pd.to_numeric(exports_long["Exports"], errors="coerce")

print(exports_long.head())

   Country  Year  Exports
0       US  1980   967.03
1       UK  1980   528.75
2  Austria  1980    10.90
3  Belgium  1980      NaN
4  Denmark  1980    34.98


<h3>4. Imports Cleaning</h3>

In [12]:
imports_long["Imports"] = imports_long["Imports"].replace("–", np.nan)

imports_long["Imports"] = pd.to_numeric(imports_long["Imports"], errors="coerce")

print(imports_long.head())

   Country  Year  Imports
0       US  1980  1865.21
1       UK  1980   953.53
2  Austria  1980    34.00
3  Belgium  1980      NaN
4  Denmark  1980    26.32


In [13]:
# Check
print(cpi_long.isna().sum())
print(exchange_long.isna().sum())
print(exports_long.isna().sum())
print(imports_long.isna().sum())

Country       0
Year          0
CPI        1231
dtype: int64
Country            0
Year               0
Exchange_Rate    415
dtype: int64
Country       0
Year          0
Exports    1966
dtype: int64
Country       0
Year          0
Imports    2707
dtype: int64


In [14]:
# Convert values to numeric (force clean)
cpi_long["CPI"] = pd.to_numeric(cpi_long["CPI"], errors='coerce')
exchange_long["Exchange_Rate"] = pd.to_numeric(exchange_long["Exchange_Rate"], errors='coerce')
exports_long["Exports"] = pd.to_numeric(exports_long["Exports"], errors='coerce')
imports_long["Imports"] = pd.to_numeric(imports_long["Imports"], errors='coerce')

In [15]:
# For CPI & Exchange Rate: Drop missing rows : Needed for formula → must be valid,So we drop missing
cpi_long = cpi_long.dropna(subset=["CPI"])
exchange_long = exchange_long.dropna(subset=["Exchange_Rate"])

In [16]:
# For Trade (Exports & Imports): Fill missing with 0 : Missing trade ≈ no trade,So we fill with 0
exports_long["Exports"] = exports_long["Exports"].fillna(0)
imports_long["Imports"] = imports_long["Imports"].fillna(0)

In [17]:
# Final Check
print(cpi_long.isna().sum())
print(exchange_long.isna().sum())
print(exports_long.isna().sum())
print(imports_long.isna().sum())

Country    0
Year       0
CPI        0
dtype: int64
Country          0
Year             0
Exchange_Rate    0
dtype: int64
Country    0
Year       0
Exports    0
dtype: int64
Country    0
Year       0
Imports    0
dtype: int64


<h4>Merge all datasets into one master table</h4>
<P>Now you’ll combine everything into one dataset like this: </P></br>
| Country | Year | CPI | Exchange_Rate | Exports | Imports |</P>

In [18]:
# Merge CPI + Exchange Rate
merged_df = pd.merge(cpi_long, exchange_long,
                     on=["Country", "Year"],
                     how="inner")

In [19]:
# Merge Exports
merged_df = pd.merge(merged_df, exports_long,
                     on=["Country", "Year"],
                     how="left")

In [20]:
# Merge Imports
merged_df = pd.merge(merged_df, imports_long,
                     on=["Country", "Year"],
                     how="left")

In [21]:
# Exports/Imports might have missing values after merge → fix again
merged_df["Exports"] = merged_df["Exports"].fillna(0)
merged_df["Imports"] = merged_df["Imports"].fillna(0)

In [22]:
# final result: merge table
print(merged_df.head())
print(merged_df.shape)

     Country  Year     CPI  Exchange_Rate  Exports  Imports
0    Algeria  1980   5.933       3.837450    10.91     0.00
1  Australia  1980  27.419       0.877501   124.23   210.20
2    Austria  1980  48.900       0.718901    10.90    34.00
3    Bahamas  1980  36.547       1.000000     0.80    10.63
4    Bahrain  1980  65.239       0.377000    22.43    88.31
(7021, 6)


<h3>Calculate Trade Weight</h3>

In [23]:
# total trade
merged_df["Trade"] = merged_df["Exports"] + merged_df["Imports"]

In [24]:
# total trade per year
total_trade_year = merged_df.groupby("Year")["Trade"].transform("sum")

In [25]:
# trade weights
merged_df["Weight"] = merged_df["Trade"] / total_trade_year

In [26]:
# Check weights sum to 1 for each year:
print(merged_df.groupby("Year")["Weight"].sum().head())

Year
1980    1.0
1981    1.0
1982    1.0
1983    1.0
1984    1.0
Name: Weight, dtype: float64


In [27]:
merged_df["Weight"] = merged_df["Weight"].fillna(0)

<h3>Compute RER (Real Exchange Rate)</h3>
<P>For each country:

RER = Exchange Rate × (India CPI / Country CPI)</P>

In [28]:
# Extract India CPI
india_cpi = cpi_long[cpi_long["Country"] == "India"][["Year", "CPI"]]
india_cpi.rename(columns={"CPI": "CPI_India"}, inplace=True)

In [29]:
# Merge India CPI into main data
merged_df = pd.merge(merged_df, india_cpi, on="Year", how="left")


In [30]:
print((merged_df["CPI"] == 0).sum())

49


In [31]:
merged_df["CPI"] = merged_df["CPI"].replace(0, np.nan)

In [32]:
# Compute RER
merged_df["RER"] = merged_df["Exchange_Rate"] * (
    merged_df["CPI_India"] / merged_df["CPI"]
)


In [33]:
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)  # remove infinite values
merged_df = merged_df.dropna(subset=["RER"])  # drop bad rows

In [34]:
# Quick Check
print(merged_df[["Country", "Year", "RER"]].head())

     Country  Year       RER
0    Algeria  1980  6.621267
1  Australia  1980  0.327619
2    Austria  1980  0.150499
3    Bahamas  1980  0.280105
4    Bahrain  1980  0.059157


<P>If India inflation ↑ → RER ↑ → currency less competitive </br>
If foreign inflation ↑ → RER ↓

This is the real adjustment part</P>

<h3>Compute REER (final index)</h3>
<p>Now combine all countries:

REER = Σ (Weight × RER)</p>

In [35]:
# Weighted RER
merged_df["Weighted_RER"] = merged_df["Weight"] * merged_df["RER"]

In [36]:
# Aggregate to REER (per year)
reer_df = merged_df.groupby("Year")["Weighted_RER"].sum().reset_index()
reer_df.rename(columns={"Weighted_RER": "REER"}, inplace=True)

print(reer_df.head())

   Year        REER
0  1980  174.052152
1  1981  241.239936
2  1982  127.523239
3  1983  125.517378
4  1984   91.778142


In [37]:
# Rebase
base_year = 2015

base_value = reer_df[reer_df["Year"] == base_year]["REER"].values[0]

reer_df["REER_Index"] = (reer_df["REER"] / base_value) * 100

In [38]:
print(reer_df.head())

   Year        REER  REER_Index
0  1980  174.052152   13.822779
1  1981  241.239936   19.158662
2  1982  127.523239   10.127571
3  1983  125.517378    9.968271
4  1984   91.778142    7.288786


In [40]:
print(reer_df.describe())

              Year         REER  REER_Index
count    41.000000    41.000000   41.000000
mean   2000.000000   602.488905   47.848135
std      11.979149   511.115253   40.591472
min    1980.000000    44.018045    3.495801
25%    1990.000000   174.052152   13.822779
50%    2000.000000   431.499379   34.268582
75%    2010.000000   842.268479   66.890818
max    2020.000000  1971.309678  156.556396


In [41]:
print(reer_df.isna().sum())

Year          0
REER          0
REER_Index    0
dtype: int64


In [46]:
reer_df = reer_df.sort_values("Year")

In [47]:
# Export csv
reer_df.to_csv("output2_Tanisha_Jain.csv", index=False)